# DiagnosticIQ source step 2: consistency check

Splits `extracted/extracted_data.json` by whether each rule has parsed conditions and observations. Rules with both are written to `extracted/observations.rules.json`, the input to the later steps. Debug listings go to `$ASSETOPSBENCH_DATASET_DIR/debug/`.

In [ ]:
import os
import re
import json
from datetime import datetime
from typing import Optional, List, Callable, Any
import copy

In [ ]:
#parameter 

from dataset_utils.paths import get_dataset_dir

dataset_dir = get_dataset_dir()
extract_fol = dataset_dir / "extracted"
debug_fol = dataset_dir / "debug"
save_loc = os.path.join(extract_fol,"extracted_data.json")
os.makedirs(extract_fol, exist_ok=True)
os.makedirs(debug_fol, exist_ok=True)


In [ ]:
def save_json(data:dict, loc:str) -> None:
    with open(loc, "w") as f0:
        json.dump(data, f0)


def text_write(text, filename= "debug.test"):
    with open(filename, "w") as f0:
        f0.write(text)

def dict_value_string_function(d:dict, func:Callable[[Optional[Any]], Optional[Any]]) -> dict:
    r_dict = {}
    for k,v in d.items():
        if isinstance(v, dict):
            r_dict[k] = dict_value_string_function(v,func)
        else:
            r_dict[k] = func(v)

    return r_dict

In [ ]:
with open(save_loc, "r") as f0:
    data = json.load(f0)

dataset = data["rule_set"]

In [ ]:
f = ""
cleaned = copy.deepcopy(data)
cleaned["rule_set"] = []
for no, s in enumerate(dataset):
    if isinstance(s["display_text"]["rules"],str) and isinstance(s["display_text"]["observations"],str):
        if s["#n"] == 0:
            print("YUp")

        f += "<><><><>\n"
        f += s["id"]+"\n\n"
        f += s["display_text"]["rules"]+"\n\n"
        f += s["display_text"]["observations"]+"\n"

        s = dict_value_string_function(s, lambda x: x.strip() if isinstance(x,str) else x)
        cleaned["rule_set"].append(s)

print(len(cleaned["rule_set"]))
text_write(f,os.path.join(debug_fol, "both.debug.test"))
save_json(cleaned, os.path.join(extract_fol,"observations.rules.json"))



In [ ]:
f = ""
cleaned = copy.deepcopy(data)
cleaned["rule_set"] = []
for no, s in enumerate(dataset):
    if (not isinstance(s["display_text"]["rules"],str)) and (not isinstance(s["display_text"]["observations"],str)):
        f += "<><><><>\n"
        f += s["id"]+"\n"

        s = dict_value_string_function(s, lambda x: x.strip() if isinstance(x,str) else x)
        cleaned["rule_set"].append(s)

print(len(cleaned["rule_set"]))
text_write(f,os.path.join(debug_fol, "both_not.debug.test"))
save_json(cleaned, os.path.join(extract_fol,"no_observations.no_rules.json"))


In [ ]:
f = ""
cleaned = copy.deepcopy(data)
cleaned["rule_set"] = []
for no, s in enumerate(dataset):
    if (not isinstance(s["display_text"]["rules"],str)) and (isinstance(s["display_text"]["observations"],str)):
        f += "<><><><>\n"
        f += s["id"]+"\n\n"
        f += s["display_text"]["observations"]+"\n"

        s = dict_value_string_function(s, lambda x: x.strip() if isinstance(x,str) else x)
        cleaned["rule_set"].append(s)

print(len(cleaned["rule_set"]))
text_write(f,os.path.join(debug_fol, "observations_only.debug.test"))
save_json(cleaned, os.path.join(extract_fol,"only_observations.json"))


In [ ]:
f = ""
cleaned = copy.deepcopy(data)
cleaned["rule_set"] = []
for no, s in enumerate(dataset):
    if (isinstance(s["display_text"]["rules"],str)) and (not isinstance(s["display_text"]["observations"],str)):
        f += "<><><><>\n"
        f += s["id"]+"\n\n"
        f += s["display_text"]["rules"]+"\n"

        s = dict_value_string_function(s, lambda x: x.strip() if isinstance(x,str) else x)
        cleaned["rule_set"].append(s)

print(len(cleaned["rule_set"]))
text_write(f,os.path.join(debug_fol, "rules_only.debug.test"))
save_json(cleaned, os.path.join(extract_fol,"only_rules.json"))
